In [1]:
import sys
from pathlib import Path
import pandas as pd

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

%load_ext autoreload
%autoreload 2

path: /workspace


In [2]:
# Extract canonical facts from FCA HTML files
from src.parsing.arelle_parser import process_html_files

html_folder = projectRoot / "data" / "raw" / "fca"
output_path = (projectRoot / "data" / "processed" / 
               "canonical_facts" / "bronze.csv")

bronze_df = process_html_files(html_folder, output_path, debug=False)

sample_bronze = bronze_df.sample(n=250, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_bronze.csv")
sample_bronze.to_csv(csv_path, index=False)
print(f"Bronze sample extract saved to: {csv_path}")

Bronze data saved to: /workspace/data/processed/canonical_facts/bronze.csv
Bronze sample extract saved to: /workspace/data/processed/debug/sample_bronze.csv


In [3]:
# Create silver ground truth from bronze data
from src.parsing.canonical_facts import create_silver_ground_truth

silver_df = create_silver_ground_truth(bronze_df, 
                        str(projectRoot / "data" / "processed" /
                            "canonical_facts" / "silver.csv"))

sample_silver = silver_df.sample(n=300, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_silver.csv")
sample_silver.to_csv(csv_path, index=False)
print(f"Silver sample extract saved to: {csv_path}")

Created cleaned ground truth with 5174 rows.
Silver data saved to: /workspace/data/processed/canonical_facts/silver.csv
Silver sample extract saved to: /workspace/data/processed/debug/sample_silver.csv


In [4]:
print(silver_df.keys())
pivot_table = silver_df.pivot_table(
    index=['entity_name', 'taxonomy_domain'],
    aggfunc='size',
    fill_value=0,
)
print(pivot_table)

Index(['filing_id', 'context_id', 'raw_name', 'taxonomy_domain', 'data_type',
       'unit_ref', 'decimals', 'value_text', 'value_numeric', 'entity_scheme',
       'entity_identifier', 'period_type', 'instant', 'period_start',
       'period_end', 'dimensional_qualifier', 'ground_truth_value',
       'period_end_dt', 'period_start_dt', 'year', 'entity_name', 'segment'],
      dtype='object')
entity_name  taxonomy_domain
AstraZeneca  azn                 415
             ifrs-full          2109
GSK          gsk                 503
             ifrs-full          2147
dtype: int64


In [4]:
# Create gold ground truth from silver data
from src.parsing.canonical_facts import create_gold_ground_truth

gold_df = create_gold_ground_truth(silver_df, 
                         str(projectRoot / "data" / "processed" /
                              "canonical_facts" / "gold.csv"))
print(gold_df.head())

sample_gold = gold_df.sample(n=300, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_gold.csv")
sample_gold.to_csv(csv_path, index=False)
print(f"Gold sample extract saved to: {csv_path}")

Gold data saved to: /workspace/data/processed/canonical_facts/gold.csv
   entity_name                 segment                  canonical_fact_name  \
0  AstraZeneca  Other_Financial_Metric  Date Of End Of Reporting Period2013   
1  AstraZeneca  Other_Financial_Metric             Country Of Incorporation   
2  AstraZeneca        Income_Statement           Revenue From Sale Of Goods   
3  AstraZeneca        Income_Statement           Revenue From Sale Of Goods   
4  AstraZeneca        Income_Statement           Revenue From Sale Of Goods   

  ground_truth_value period_type  year  
0         2022-12-31    duration  2023  
1  England and Wales    duration  2023  
2      42998000000.0    duration  2023  
3      36541000000.0    duration  2022  
4      25890000000.0    duration  2021  
Gold sample extract saved to: /workspace/data/processed/debug/sample_gold.csv
